# Search Results Crawler

Crawls Amazon search result pages for a list of keywords, extracts product cards (ASIN, title, brand, price, rating, reviews, sponsored flag) and enriches them with price-history data from a third-party price tracker.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


> ⚠️ **Read before running.** This notebook sends automated requests to aiprice.com to pull price history for each product. That site's terms of service may prohibit automated access.
>
> It is shared to show the collection and parsing method, not as a tool to run against aiprice.com (a third-party price tracker). Do not run it against the live site without that site's permission. To follow the downstream steps, use the synthetic files in [`sample-data/`](../sample-data/).

**Cell 01**

In [ ]:
# ================================================================================
# CELL 1. Amazon 검색결과 HTML 수집 + 상품정보 추출
#
# 수정사항
#   1. Amazon HTML만 처리
#   2. Sephora HTML 자동 제외
#   3. Amazon URL의 page=2/3/4 정확히 인식
#   4. 파일명에서 검색키워드 정확히 추출
#   5. 기존 정상 작동하던 상품 블록 추출 구조 유지
#   6. 브랜드 추출만 개선
# ================================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import json
import html as html_lib
import glob
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote


# ================================================================================
# 0. 경로 / 설정
# ================================================================================

HTML_DIR = (
    "/content/drive/MyDrive/"
    "YOUR_PROJECT_FOLDER/amazon_data/"
    "amazon_search_html_260922/eye+patch"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "YOUR_PROJECT_FOLDER/amazon_data/"
    "amazon_output/"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPECTED_PER_PAGE = 52


# ================================================================================
# 1. Amazon HTML 파일만 찾기
# ================================================================================

def find_amazon_html_files(html_dir):

    patterns = [
        os.path.join(html_dir, "*.html"),
        os.path.join(html_dir, "*.htm"),
    ]

    all_files = []

    for pattern in patterns:
        all_files.extend(glob.glob(pattern))

    # 중복 제거
    all_files = list(set(all_files))

    # ------------------------------------------------------------
    # Amazon 파일만 남김
    #
    # 파일명에:
    #   amazon.com
    #
    # 이 포함되어 있는 경우만 처리
    # ------------------------------------------------------------

    amazon_files = []

    for path in all_files:

        filename = os.path.basename(path).lower()

        if "amazon.com" in filename:
            amazon_files.append(path)

    # 파일명 정렬
    amazon_files = sorted(
        amazon_files,
        key=lambda x: os.path.basename(x).lower()
    )

    return amazon_files


amazon_files = find_amazon_html_files(
    HTML_DIR
)


print("=" * 80)
print("Amazon HTML 파일 검색")
print("=" * 80)

print(f"HTML 폴더 : {HTML_DIR}")
print(f"Amazon 파일 수 : {len(amazon_files)}")

for i, path in enumerate(amazon_files, 1):

    print(
        f"{i:02d}. "
        f"{os.path.basename(path)}"
    )


# ================================================================================
# 2. 파일명에서 페이지 번호 추출
# ================================================================================

def extract_page_number_from_filename(filename):

    """
    Amazon 파일명에서 페이지 번호 추출.

    예:

    ...&page=2&...
        -> 2

    ...&page=3&...
        -> 3

    ...&page=4&...
        -> 4

    page 파라미터가 없는 경우
        -> 1
    """

    name = os.path.basename(filename)

    # ------------------------------------------------------------
    # URL 형식의 page=
    # ------------------------------------------------------------

    m = re.search(
        r"[?&]page=(\d+)",
        name,
        flags=re.I
    )

    if m:

        try:
            return int(m.group(1))
        except Exception:
            pass

    # ------------------------------------------------------------
    # 파일명에 | page 2 등이 있는 경우
    # ------------------------------------------------------------

    m = re.search(
        r"\|\s*page\s*(\d+)",
        name,
        flags=re.I
    )

    if m:

        try:
            return int(m.group(1))
        except Exception:
            pass

    # ------------------------------------------------------------
    # 일반 page2 / page_2
    # ------------------------------------------------------------

    m = re.search(
        r"page[\s_-]*(\d+)",
        name,
        flags=re.I
    )

    if m:

        try:
            return int(m.group(1))
        except Exception:
            pass

    return 1


# ================================================================================
# 3. 파일명에서 검색키워드 추출
# ================================================================================

def extract_keyword_from_filename(filename):

    """
    Amazon URL 형태의 파일명에서
    s_k= 뒤의 검색어를 추출한다.

    예:
        ...amazon.com_s_k=acne+patch&s=...
        -> acne patch
    """

    name = os.path.basename(filename)

    # ------------------------------------------------------------
    # amazon.com_s_k= 형태
    # ------------------------------------------------------------

    m = re.search(
        r"amazon\.com_s_k=([^&]+)",
        name,
        flags=re.I
    )

    if m:

        keyword = m.group(1)

        keyword = unquote(
            keyword
        )

        keyword = keyword.replace(
            "+",
            " "
        )

        keyword = re.sub(
            r"\s+",
            " ",
            keyword
        ).strip()

        if keyword:
            return keyword

    # ------------------------------------------------------------
    # 일반 URL 형태의 ?k=
    # ------------------------------------------------------------

    m = re.search(
        r"(?:[?&]|_s_)k=([^&]+)",
        name,
        flags=re.I
    )

    if m:

        keyword = unquote(
            m.group(1)
        )

        keyword = keyword.replace(
            "+",
            " "
        )

        keyword = re.sub(
            r"\s+",
            " ",
            keyword
        ).strip()

        if keyword:
            return keyword

    # ------------------------------------------------------------
    # 기존 | page 형식 보조
    # ------------------------------------------------------------

    base = os.path.splitext(
        name
    )[0]

    base = re.sub(
        r"\s*\|\s*page\s*\d+\s*$",
        "",
        base,
        flags=re.I
    )

    return base.strip()


# ================================================================================
# 4. view-source HTML 복원
# ================================================================================
# ================================================================================
# View Source HTML → 실제 Amazon HTML 복원 v2
#
# Chrome에서 저장한 view-source 페이지는
# 실제 HTML이 <td class="line-content"> 안의
# html-tag / html-attribute-name / html-attribute-value span으로 분해되어 있음.
#
# 따라서 line-content의 화면 텍스트를 다시 합쳐서
# 실제 Amazon HTML로 복원한다.
# ================================================================================

from bs4 import BeautifulSoup
from html import unescape


def restore_view_source_html(raw_html):

    if not raw_html:
        return ""

    # --------------------------------------------------------------------------
    # 1. Chrome View Source wrapper인지 확인
    # --------------------------------------------------------------------------

    if (
        "line-content" not in raw_html
        and "html-tag" not in raw_html
        and "line-gutter-backdrop" not in raw_html
    ):
        # 이미 일반 HTML이면 그대로 반환
        return raw_html

    # --------------------------------------------------------------------------
    # 2. View Source HTML 파싱
    # --------------------------------------------------------------------------

    try:

        source_soup = BeautifulSoup(
            raw_html,
            "html.parser"
        )

    except Exception:

        return raw_html

    # --------------------------------------------------------------------------
    # 3. 실제 source line 추출
    #
    # Chrome View Source:
    #
    # <td class="line-content">
    #     <span class="html-tag">...</span>
    # </td>
    #
    # 각 span의 text를 합치면 원래 HTML 코드가 복원된다.
    # --------------------------------------------------------------------------

    line_nodes = source_soup.select(
        "td.line-content"
    )

    if not line_nodes:

        return raw_html

    lines = []

    for node in line_nodes:

        # 화면에 표시되는 실제 source code
        text = node.get_text(
            "",
            strip=False
        )

        lines.append(text)

    restored = "\n".join(lines)

    # --------------------------------------------------------------------------
    # 4. HTML entity 복원
    #
    # &quot; → "
    # &amp;   → &
    # &lt;    → <
    # &gt;    → >
    # --------------------------------------------------------------------------

    restored = unescape(restored)

    # --------------------------------------------------------------------------
    # 5. Chrome View Source에서 남을 수 있는 불필요한 wrapper 제거
    # --------------------------------------------------------------------------

    restored = restored.strip()

    return restored
# ================================================================================
# 5. URL에서 검색어 추출
# ================================================================================

def extract_keyword_from_url(url):

    if not url:
        return ""

    try:

        parsed = urlparse(
            url
        )

        query = parse_qs(
            parsed.query
        )

        for key in [
            "q",
            "k",
            "field-keywords",
            "keywords"
        ]:

            if key in query and query[key]:

                value = unquote(
                    str(query[key][0])
                )

                value = value.replace(
                    "+",
                    " "
                )

                value = re.sub(
                    r"\s+",
                    " ",
                    value
                ).strip()

                if value:
                    return value

    except Exception:
        pass

    return ""


# ================================================================================
# 6. ASIN 추출
# ================================================================================

def extract_asin(node):

    # ------------------------------------------------------------
    # 1. data-asin
    # ------------------------------------------------------------

    try:

        asin = node.get(
            "data-asin",
            ""
        )

        if asin:

            asin = str(
                asin
            ).strip().upper()

            if re.fullmatch(
                r"[A-Z0-9]{10}",
                asin
            ):
                return asin

    except Exception:
        pass

    # ------------------------------------------------------------
    # 2. HTML 내부 Amazon URL
    # ------------------------------------------------------------

    try:

        for a in node.find_all(
            "a",
            href=True
        ):

            href = html_lib.unescape(
                str(a.get("href"))
            )

            m = re.search(
                r"/(?:dp|gp/product|product)/([A-Z0-9]{10})",
                href,
                flags=re.I
            )

            if m:

                return m.group(1).upper()

    except Exception:
        pass

    # ------------------------------------------------------------
    # 3. 텍스트 / attribute 내부
    # ------------------------------------------------------------

    try:

        node_text = str(
            node
        )

        m = re.search(
            r"/(?:dp|gp/product|product)/([A-Z0-9]{10})",
            node_text,
            flags=re.I
        )

        if m:
            return m.group(1).upper()

    except Exception:
        pass

    return ""


# ================================================================================
# 7. 상품명 추출
# ================================================================================

def extract_title(node):

    selectors = [
        "h2 a span",
        "h2 span",
        "h2 a",
        "h2",
    ]

    for selector in selectors:

        try:

            elements = node.select(
                selector
            )

        except Exception:

            elements = []

        for el in elements:

            text = el.get_text(
                " ",
                strip=True
            )

            text = re.sub(
                r"\s+",
                " ",
                text
            ).strip()

            if text and len(text) >= 3:

                return text

    # ------------------------------------------------------------
    # data-cy title
    # ------------------------------------------------------------

    try:

        elements = node.select(
            '[data-cy="title-recipe"]'
        )

        for el in elements:

            text = el.get_text(
                " ",
                strip=True
            )

            text = re.sub(
                r"\s+",
                " ",
                text
            ).strip()

            if text:
                return text

    except Exception:
        pass

    return ""


# ================================================================================
# 8. 상품 URL 추출
# ================================================================================

def extract_product_url(
    node,
    asin=None
):

    if asin:

        return (
            "https://www.amazon.com/dp/"
            f"{asin}"
        )

    try:

        for a in node.find_all(
            "a",
            href=True
        ):

            href = html_lib.unescape(
                str(a.get("href"))
            )

            m = re.search(
                r"/(?:dp|gp/product|product)/([A-Z0-9]{10})",
                href,
                flags=re.I
            )

            if m:

                found_asin = (
                    m.group(1)
                    .upper()
                )

                return (
                    "https://www.amazon.com/dp/"
                    f"{found_asin}"
                )

    except Exception:
        pass

    return ""


# ================================================================================
# 9. 숫자 파싱
# ================================================================================

def parse_amazon_number(text):

    if text is None:
        return np.nan

    text = str(
        text
    ).strip()

    if not text:
        return np.nan

    text = text.replace(
        ",",
        ""
    )

    m = re.search(
        r"-?\d+(?:\.\d+)?",
        text
    )

    if not m:
        return np.nan

    try:
        return float(
            m.group(0)
        )
    except Exception:
        return np.nan


# ================================================================================
# 10. 가격 파싱
# ================================================================================

def parse_price_text(text):

    if text is None:
        return np.nan

    text = str(
        text
    ).strip()

    if not text:
        return np.nan

    matches = re.findall(
        r"\$?\s*(\d+(?:,\d{3})*(?:\.\d{1,2})?)",
        text
    )

    if not matches:
        return np.nan

    try:

        return float(
            matches[0].replace(
                ",",
                ""
            )
        )

    except Exception:

        return np.nan


# ================================================================================
# 11. 가격 추출
# ================================================================================

def extract_prices(node):

    sale_price = np.nan
    list_price = np.nan
    discount_rate = np.nan

    # ------------------------------------------------------------
    # 판매가격
    # ------------------------------------------------------------

    sale_selectors = [
        "span.a-price > span.a-offscreen",
        "span.a-price span.a-offscreen",
        ".a-price .a-offscreen",
    ]

    sale_candidates = []

    for selector in sale_selectors:

        try:
            elements = node.select(
                selector
            )
        except Exception:
            elements = []

        for el in elements:

            text = el.get_text(
                " ",
                strip=True
            )

            price = parse_price_text(
                text
            )

            if not pd.isna(price):

                sale_candidates.append(
                    price
                )

    if sale_candidates:

        sale_price = sale_candidates[0]

    # ------------------------------------------------------------
    # 정가
    # ------------------------------------------------------------

    list_selectors = [
        ".a-text-price span.a-offscreen",
        "span.a-text-price span.a-offscreen",
    ]

    list_candidates = []

    for selector in list_selectors:

        try:
            elements = node.select(
                selector
            )
        except Exception:
            elements = []

        for el in elements:

            text = el.get_text(
                " ",
                strip=True
            )

            price = parse_price_text(
                text
            )

            if not pd.isna(price):

                list_candidates.append(
                    price
                )

    if list_candidates:

        list_price = list_candidates[0]

    # ------------------------------------------------------------
    # 할인율
    # ------------------------------------------------------------

    try:

        text = node.get_text(
            " ",
            strip=True
        )

        m = re.search(
            r"(\d+)\s*%\s*off",
            text,
            flags=re.I
        )

        if m:

            discount_rate = float(
                m.group(1)
            )

    except Exception:
        pass

    # ------------------------------------------------------------
    # 정가 역산
    # ------------------------------------------------------------

    if (
        pd.isna(list_price)
        and not pd.isna(sale_price)
        and not pd.isna(discount_rate)
        and discount_rate < 100
    ):

        try:

            list_price = round(
                sale_price
                / (1 - discount_rate / 100),
                2
            )

        except Exception:
            pass

    # ------------------------------------------------------------
    # 할인율 계산
    # ------------------------------------------------------------

    if (
        pd.isna(discount_rate)
        and not pd.isna(sale_price)
        and not pd.isna(list_price)
        and list_price > 0
        and list_price > sale_price
    ):

        discount_rate = round(
            (
                1
                - sale_price / list_price
            ) * 100,
            1
        )

    return (
        sale_price,
        list_price,
        discount_rate
    )


# ================================================================================
# 12. 평점 추출
# ================================================================================

def extract_rating(node):

    selectors = [
        "span.a-icon-alt",
        "[aria-label*='out of 5']",
    ]

    for selector in selectors:

        try:
            elements = node.select(
                selector
            )
        except Exception:
            elements = []

        for el in elements:

            text = (
                el.get("aria-label")
                or el.get_text(
                    " ",
                    strip=True
                )
            )

            m = re.search(
                r"(\d+(?:\.\d+)?)\s+out\s+of\s+5",
                text,
                flags=re.I
            )

            if m:

                try:
                    return float(
                        m.group(1)
                    )
                except Exception:
                    pass

    return np.nan


# ================================================================================
# 13. 리뷰수 추출
# ================================================================================

def extract_review_count(node):

    selectors = [
        "a[aria-label*='ratings']",
        "a[aria-label*='rating']",
        "span.a-size-base.s-underline-text",
    ]

    for selector in selectors:

        try:
            elements = node.select(
                selector
            )
        except Exception:
            elements = []

        for el in elements:

            text = (
                el.get("aria-label")
                or el.get_text(
                    " ",
                    strip=True
                )
            )

            m = re.search(
                r"([\d,]+)\s*(?:ratings?|reviews?)",
                text,
                flags=re.I
            )

            if m:

                try:

                    return int(
                        m.group(1).replace(
                            ",",
                            ""
                        )
                    )

                except Exception:
                    pass

            m = re.fullmatch(
                r"\s*([\d,]+)\s*",
                text
            )

            if m:

                try:

                    return int(
                        m.group(1).replace(
                            ",",
                            ""
                        )
                    )

                except Exception:
                    pass

    return np.nan


# ================================================================================
# 14. 브랜드 후보 검증
# ================================================================================

def _is_valid_amazon_brand(value):

    if value is None:
        return False

    value = re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()

    if not value:
        return False

    if len(value) > 100:
        return False

    # 숫자만
    if re.fullmatch(
        r"[\d\s.,$%]+",
        value
    ):
        return False

    # Amazon UI / 배지 / 구매 관련 텍스트
    invalid_patterns = [

        r"^join\s+prime$",
        r"^prime$",
        r"^sponsored$",
        r"^climate\s+pledge\s+friendly$",
        r"^limited\s+time\s+deal$",
        r"^deal$",
        r"^coupon$",
        r"^subscribe\s*&\s*save$",
        r"^buy\s+with\s+prime$",
        r"^amazon'?s\s+choice$",
        r"^overall\s+pick$",
        r"^best\s+seller$",
        r"^best\s+seller\s+in.*$",
        r"^free\s+shipping$",
        r"^free\s+delivery$",
        r"^in\s+cart$",
        r"^add\s+to\s+cart$",
        r"^currently\s+unavailable$",
        r"^see\s+options$",
        r"^see\s+all\s+options$",
        r"^more\s+buying\s+choices$",
        r"^new$",
        r"^used$",
        r"^refurbished$",
        r"^renewed$",
    ]

    for pattern in invalid_patterns:

        if re.match(
            pattern,
            value,
            flags=re.I
        ):
            return False

    # 가격
    if re.fullmatch(
        r"\$?\s*\d+(?:\.\d+)?",
        value
    ):
        return False

    # 평점
    if re.search(
        r"^\d+(?:\.\d+)?\s*(?:out of 5|stars?)$",
        value,
        flags=re.I
    ):
        return False

    # 리뷰수
    if re.search(
        r"^[\d,]+\+?\s*(?:ratings?|reviews?)$",
        value,
        flags=re.I
    ):
        return False

    return True

# ================================================================================
# 14-1. 브랜드 문자열 정리 / 검증 helper
# ================================================================================

def _clean_brand_value(value):

    if value is None:
        return ""

    value = html_lib.unescape(
        str(value)
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    # 앞뒤 불필요한 따옴표 / 구분자 제거
    value = value.strip(
        " \t\r\n.,:;|/\\()[]{}<>\"'“”‘’"
    )

    return value


def _is_invalid_brand(value):

    if value is None:
        return True

    value = _clean_brand_value(
        value
    )

    if not value:
        return True

    # 기존 Amazon 브랜드 검증 함수 사용
    return not _is_valid_amazon_brand(
        value
    )


# ================================================================================
# 15. 브랜드 추출
# ================================================================================
def extract_brand(block, title=None):

    # =========================================================================
    # 1. 명시적인 HTML brand metadata가 있는 경우
    # =========================================================================

    if block is not None:

        try:
            nodes = block.select('[itemprop="brand"]')
        except Exception:
            nodes = []

        for node in nodes:

            value = (
                node.get("content")
                or node.get_text(" ", strip=True)
            )

            value = _clean_brand_value(value)

            if value and not _is_invalid_brand(value):
                return value

        # data-* attribute
        try:
            tags = block.find_all(True)
        except Exception:
            tags = []

        for tag in tags:

            attrs = getattr(tag, "attrs", {}) or {}

            for attr_name, attr_value in attrs.items():

                attr_lower = str(attr_name).lower()

                if (
                    "brand" not in attr_lower
                    and "manufacturer" not in attr_lower
                ):
                    continue

                if isinstance(attr_value, list):
                    value = " ".join(map(str, attr_value))
                else:
                    value = str(attr_value)

                value = _clean_brand_value(value)

                if value and not _is_invalid_brand(value):
                    return value

    # =========================================================================
    # 2. 상품명 확보
    # =========================================================================

    if not title:
        try:
            title = extract_title(block)
        except Exception:
            title = ""

    if not title:
        return ""

    title = re.sub(r"\s+", " ", str(title)).strip()

    # =========================================================================
    # 3. 상품명 앞쪽 정리
    # =========================================================================

    title_for_brand = re.sub(
        r"^\s*\d+(?:\.\d+)?\s*"
        r"(?:pcs?|pieces?|ct|count|counts|pairs?|pair|pack|packs?)"
        r"\b[\s:/-]*",
        "",
        title,
        flags=re.I,
    )

    first_segment = re.split(
        r"\s+[—–-]\s+|,|\(",
        title_for_brand,
        maxsplit=1,
    )[0].strip()

    if not first_segment:
        return ""

    words = first_segment.split()

    if not words:
        return ""

    # =========================================================================
    # 4. 첫 단어를 브랜드 후보로 사용
    # =========================================================================

    first = words[0].strip(
        ".,:;()[]{}<>\"'“”‘’"
    )

    if not first:
        return ""

    if re.match(r"^\d", first):
        return ""

    generic_words = {
        "silicone",
        "hydrocolloid",
        "pimple",
        "acne",
        "patch",
        "patches",
        "scar",
        "tape",
        "roll",
        "eye",
        "under",
        "medical",
        "fungal",
        "motion",
        "pain",
        "natural",
        "gold",
        "star",
        "tiger",
        "wound",
        "transparent",
        "clear",
        "anti-acne",
        "fungus",
    }

    if first.lower() in generic_words:
        return ""

    candidate = first

    # =========================================================================
    # 5. 2~3단어 브랜드
    # =========================================================================

    if len(words) >= 2:

        second = words[1].strip(
            ".,:;()[]{}<>\"'“”‘’"
        )

        # grace & stella
        if second == "&" and len(words) >= 3:

            third = words[2].strip(
                ".,:;()[]{}<>\"'“”‘’"
            )

            if third:
                candidate = f"{first} & {third}"

        # Dr. Melaxin
        elif first.lower() in {"dr.", "dr"}:

            if second:
                candidate = f"{first} {second}"

    candidate = _clean_brand_value(candidate)

    if _is_invalid_brand(candidate):
        return ""

    return candidate
# ================================================================================
# 16. 실제 Amazon Product Block 추출 v2
#
# 기존 정상 작동 방식 유지
# + Amazon HTML 구조 변화 대응
# + selector 단계별 fallback
# ================================================================================

def extract_product_blocks(soup):

    blocks = []

    # ============================================================================
    # 1. 기존 정상 작동 방식
    # ============================================================================

    try:

        candidates = soup.select(
            'div[data-component-type="s-search-result"]'
        )

    except Exception:

        candidates = []

    for block in candidates:

        asin = extract_asin(block)

        if not asin:
            continue

        title = extract_title(block)

        if not title:
            continue

        blocks.append(block)

    # ============================================================================
    # 2. 기존 data-asin 방식
    # ============================================================================

    if not blocks:

        try:

            candidates = soup.select(
                "div[data-asin]"
            )

        except Exception:

            candidates = []

        for block in candidates:

            asin = extract_asin(block)

            if not asin:
                continue

            title = extract_title(block)

            if not title:
                continue

            blocks.append(block)

    # ============================================================================
    # 3. Amazon HTML의 다른 상품 container fallback
    # ============================================================================

    if not blocks:

        selectors = [

            # 일반적인 Amazon search result
            'div[data-asin]:not([data-asin=""])',

            # role 기반
            'div[role="listitem"][data-asin]',

            # s-result-item 계열
            'div.s-result-item[data-asin]',

            # Amazon product result 계열
            '[data-index][data-asin]',

        ]

        for selector in selectors:

            try:

                candidates = soup.select(selector)

            except Exception:

                candidates = []

            for block in candidates:

                asin = extract_asin(block)

                if not asin:
                    continue

                title = extract_title(block)

                if not title:
                    continue

                blocks.append(block)

            if blocks:

                break

    # ============================================================================
    # 4. ASIN이 있는 element를 직접 탐색하는 최종 fallback
    # ============================================================================

    if not blocks:

        try:

            candidates = soup.find_all(
                attrs={"data-asin": True}
            )

        except Exception:

            candidates = []

        for block in candidates:

            asin = extract_asin(block)

            if not asin:
                continue

            title = extract_title(block)

            if not title:
                continue

            blocks.append(block)

    # ============================================================================
    # 5. 중복 제거
    # ============================================================================

    unique_blocks = []

    seen_asins = set()

    for block in blocks:

        asin = extract_asin(block)

        if not asin:
            continue

        if asin in seen_asins:
            continue

        seen_asins.add(asin)

        unique_blocks.append(block)

    return unique_blocks
# ================================================================================
# 17. Amazon 파일 1개 추출
# ================================================================================

def extract_amazon_file(file_path):

    print("\n" + "=" * 80)

    print(
        f"파일 처리 : "
        f"{os.path.basename(file_path)}"
    )

    print("=" * 80)

    # ------------------------------------------------------------
    # 페이지 / 검색어
    # ------------------------------------------------------------

    page_number = (
        extract_page_number_from_filename(
            file_path
        )
    )

    filename_keyword = (
        extract_keyword_from_filename(
            file_path
        )
    )

    print(
        f"페이지     : {page_number}"
    )

    print(
        f"파일 키워드: {filename_keyword}"
    )

    # ------------------------------------------------------------
    # HTML 읽기
    # ------------------------------------------------------------

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        raw_html = f.read()

    html_text = (
        restore_view_source_html(
            raw_html
        )
    )

    print(
        f"HTML 문자 수 : {len(html_text):,}"
    )

    # ------------------------------------------------------------
    # BeautifulSoup
    # ------------------------------------------------------------

    soup = BeautifulSoup(
        html_text,
        "html.parser"
    )

    # ------------------------------------------------------------
    # Product block
    # ------------------------------------------------------------

    blocks = extract_product_blocks(
        soup
    )

    print(
        f"상품 블록 수 : {len(blocks)}"
    )

    rows = []

    # ------------------------------------------------------------
    # 상품별 추출
    # ------------------------------------------------------------

    for local_rank, block in enumerate(
        blocks,
        start=1
    ):

        asin = extract_asin(
            block
        )

        title = extract_title(
            block
        )

        product_url = extract_product_url(
            block,
            asin=asin
        )

        brand = extract_brand(
            block,
            title=title
        )

        (
            sale_price,
            list_price,
            discount_rate
        ) = extract_prices(
            block
        )

        rating = extract_rating(
            block
        )

        review_count = extract_review_count(
            block
        )

        # --------------------------------------------------------
        # 전체 검색순위
        #
        # page 1 = 1~52
        # page 2 = 53~104
        # page 3 = 105~156
        # page 4 = 157~208
        # --------------------------------------------------------

        search_rank = (
            (page_number - 1)
            * EXPECTED_PER_PAGE
            + local_rank
        )

        # --------------------------------------------------------
        # 검색키워드
        # --------------------------------------------------------

        keyword = filename_keyword

        # --------------------------------------------------------
        # 혹시 파일명에서 못 가져온 경우 URL 보조
        # --------------------------------------------------------

        if not keyword:

            try:

                # HTML의 canonical URL
                canonical = soup.select_one(
                    'link[rel="canonical"]'
                )

                if canonical:

                    href = canonical.get(
                        "href",
                        ""
                    )

                    keyword_from_url = (
                        extract_keyword_from_url(
                            href
                        )
                    )

                    if keyword_from_url:
                        keyword = keyword_from_url

            except Exception:
                pass

        rows.append({

            "검색순위": search_rank,

            "검색키워드": keyword,

            "브랜드": brand,

            "상품명": title,

            "상품URL": product_url,

            "ASIN": asin,

            "판매가격": sale_price,

            "정가": list_price,

            "할인율": discount_rate,

            "평점": rating,

            "리뷰수": review_count,

            "페이지": page_number,

        })

    # ------------------------------------------------------------
    # DataFrame
    # ------------------------------------------------------------

    df = pd.DataFrame(
        rows
    )

    columns = [
        "검색순위",
        "검색키워드",
        "브랜드",
        "상품명",
        "상품URL",
        "ASIN",
        "판매가격",
        "정가",
        "할인율",
        "평점",
        "리뷰수",
        "페이지",
    ]

    for col in columns:

        if col not in df.columns:

            df[col] = np.nan

    df = df[
        columns
    ]

    return df


# ================================================================================
# 18. 전체 Amazon HTML 추출
# ================================================================================

all_product_dfs = []

print("\n")
print("=" * 80)
print("Amazon 전체 HTML 추출 시작")
print("=" * 80)

for file_path in amazon_files:

    try:

        df_file = extract_amazon_file(
            file_path
        )

        if not df_file.empty:

            all_product_dfs.append(
                df_file
            )

    except Exception as e:

        print(
            f"❌ 파일 처리 실패 : "
            f"{os.path.basename(file_path)}"
        )

        print(
            f"   오류 : "
            f"{type(e).__name__}: {e}"
        )


# ================================================================================
# 19. 전체 DataFrame
# ================================================================================

if all_product_dfs:

    product_df = pd.concat(
        all_product_dfs,
        ignore_index=True
    )

else:

    product_df = pd.DataFrame(
        columns=[
            "검색순위",
            "검색키워드",
            "브랜드",
            "상품명",
            "상품URL",
            "ASIN",
            "판매가격",
            "정가",
            "할인율",
            "평점",
            "리뷰수",
            "페이지",
        ]
    )


# ================================================================================
# 20. 데이터 타입 정리
# ================================================================================

if not product_df.empty:

    numeric_columns = [
        "검색순위",
        "판매가격",
        "정가",
        "할인율",
        "평점",
        "리뷰수",
        "페이지",
    ]

    for col in numeric_columns:

        product_df[col] = pd.to_numeric(
            product_df[col],
            errors="coerce"
        )

    for col in [
        "검색키워드",
        "브랜드",
        "상품명",
        "상품URL",
        "ASIN",
    ]:

        product_df[col] = (
            product_df[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    product_df["ASIN"] = (
        product_df["ASIN"]
        .str.upper()
    )


# ================================================================================
# 21. 정렬
# ================================================================================

if not product_df.empty:

    product_df = (
        product_df
        .sort_values(
            by=[
                "검색키워드",
                "페이지",
                "검색순위",
            ],
            ascending=[
                True,
                True,
                True,
            ],
            kind="stable"
        )
        .reset_index(
            drop=True
        )
    )


# ================================================================================
# 22. 전체 QA
# ================================================================================

print("\n")
print("=" * 80)
print("Amazon 추출 결과 QA")
print("=" * 80)

print(
    f"전체 상품 수 : "
    f"{len(product_df):,}"
)

if not product_df.empty:

    qa_columns = [
        "검색순위",
        "검색키워드",
        "브랜드",
        "상품명",
        "상품URL",
        "ASIN",
        "판매가격",
        "정가",
        "할인율",
        "평점",
        "리뷰수",
        "페이지",
    ]

    for col in qa_columns:

        non_empty = (
            product_df[col]
            .notna()
            &
            product_df[col]
            .astype(str)
            .str.strip()
            .ne("")
        )

        count = int(
            non_empty.sum()
        )

        pct = (
            count
            / len(product_df)
            * 100
        )

        print(
            f"{col:<15} : "
            f"{count:,} / "
            f"{len(product_df):,} "
            f"({pct:.1f}%)"
        )


# ================================================================================
# 23. 페이지 QA
# ================================================================================

print("\n")
print("=" * 80)
print("페이지별 상품 수 QA")
print("=" * 80)

if not product_df.empty:

    page_summary = (
        product_df
        .groupby(
            [
                "검색키워드",
                "페이지"
            ],
            dropna=False
        )
        .size()
        .reset_index(
            name="상품수"
        )
        .sort_values(
            [
                "검색키워드",
                "페이지"
            ]
        )
    )

    display(
        page_summary
    )


# ================================================================================
# 24. 검색키워드 QA
# ================================================================================

print("\n")
print("=" * 80)
print("검색키워드별 상품 수")
print("=" * 80)

if not product_df.empty:

    keyword_summary = (
        product_df
        .groupby(
            "검색키워드",
            dropna=False
        )
        .size()
        .reset_index(
            name="상품수"
        )
        .sort_values(
            "상품수",
            ascending=False
        )
    )

    display(
        keyword_summary
    )


# ================================================================================
# 25. 브랜드 QA
# ================================================================================

print("\n")
print("=" * 80)
print("브랜드 추출 QA")
print("=" * 80)

if not product_df.empty:

    brand_success = (
        product_df["브랜드"]
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    brand_fail = (
        product_df["브랜드"]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(
        f"브랜드 추출 성공 : "
        f"{brand_success:,}"
    )

    print(
        f"브랜드 추출 실패 : "
        f"{brand_fail:,}"
    )

    print("\n브랜드 상위 30개:")

    brand_counts = (
        product_df["브랜드"]
        .replace(
            "",
            np.nan
        )
        .dropna()
        .value_counts()
        .head(30)
    )

    if not brand_counts.empty:

        print(
            brand_counts.to_string()
        )

    else:

        print(
            "브랜드 데이터 없음"
        )


# ================================================================================
# 26. 브랜드 이상값 QA
# ================================================================================

print("\n")
print("=" * 80)
print("브랜드 이상값 QA")
print("=" * 80)

if not product_df.empty:

    suspicious_patterns = [
        r"^join\s+prime$",
        r"^prime$",
        r"^sponsored$",
        r"^climate\s+pledge\s+friendly$",
        r"^limited\s+time\s+deal$",
        r"^coupon$",
        r"^subscribe\s*&\s*save$",
        r"^amazon'?s\s+choice$",
        r"^best\s+seller$",
    ]

    suspicious_regex = "|".join(
        f"(?:{p})"
        for p in suspicious_patterns
    )

    suspicious_df = product_df[
        product_df["브랜드"]
        .astype(str)
        .str.strip()
        .str.contains(
            suspicious_regex,
            case=False,
            regex=True,
            na=False
        )
    ].copy()

    print(
        f"의심 브랜드 행 수 : "
        f"{len(suspicious_df):,}"
    )

    if not suspicious_df.empty:

        display(
            suspicious_df[
                [
                    "검색순위",
                    "검색키워드",
                    "브랜드",
                    "상품명",
                    "ASIN",
                ]
            ].head(30)
        )

    else:

        print(
            "✓ Join Prime / Sponsored 등 "
            "대표적인 UI 텍스트가 브랜드로 잡힌 행이 없습니다."
        )


# ================================================================================
# 27. 최종 샘플
# ================================================================================

print("\n")
print("=" * 80)
print("Amazon 상품 추출 샘플")
print("=" * 80)

if not product_df.empty:

    display(
        product_df[
            [
                "검색순위",
                "검색키워드",
                "브랜드",
                "상품명",
                "상품URL",
                "ASIN",
                "판매가격",
                "정가",
                "할인율",
                "평점",
                "리뷰수",
                "페이지",
            ]
        ].head(20)
    )

else:

    print(
        "추출된 상품이 없습니다."
    )


# ================================================================================
# 28. 완료
# ================================================================================

print("\n")
print("=" * 80)
print("CELL 1 완료")
print("=" * 80)

print(
    f"Amazon HTML files : "
    f"{len(amazon_files)}"
)

print(
    f"product_df shape  : "
    f"{product_df.shape}"
)

print(
    f"Total products    : "
    f"{len(product_df):,}"
)

print(
    "\nCell 2에서 product_df를 그대로 사용하면 됩니다."
)

**Cell 02**

In [ ]:
# ================================================================================
# ② Amazon → AiPrice 가격이력 수집
#
# 실제 AiPrice Amazon 구조:
#
# Amazon 상품 URL
#   ↓
# /Search/amazon.html?link=...
#   ↓
# AiPrice 내부 sku_id
#   ↓
# POST /Search/priceTracking.html
#   ↓
# adid=23 / currency=USD / day=180
#   ↓
# JSON 가격이력
#
# 기존 Coupang용 AiPrice 로직과 완전히 분리
# ================================================================================

import requests
import re
import json
import time
import random
import pandas as pd

from bs4 import BeautifulSoup
from datetime import datetime


# ================================================================================
# 0. AiPrice 기본 설정
# ================================================================================

AIPRICE_BASE_URL = "https://www.aiprice.com"

AIPRICE_AMAZON_URL = (
    "https://www.aiprice.com/Search/amazon.html"
)

AIPRICE_HISTORY_URL = (
    "https://www.aiprice.com/Search/priceTracking.html"
)

# Amazon에서 실제 브라우저 요청으로 확인된 값
AIPRICE_ADID = "23"
AIPRICE_CURRENCY = "USD"
AIPRICE_DAY = "180"

# 요청 간격
AIPRICE_SLEEP_MIN = 0.8
AIPRICE_SLEEP_MAX = 1.5


# ================================================================================
# 1. Session
# ================================================================================

aiprice_session = requests.Session()

aiprice_session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36 Edg/153.0.0.0"
    ),

    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),

    "Accept-Language": (
        "ko,en;q=0.9,en-US;q=0.8"
    ),
})


# ================================================================================
# 2. Amazon URL 정리
# ================================================================================

def normalize_amazon_url(url):

    if not url:
        return ""

    url = str(url).strip()

    if not url:
        return ""

    # HTML entity
    url = (
        url
        .replace("&amp;", "&")
        .replace("&quot;", '"')
    )

    # 앞뒤 따옴표 제거
    url = url.strip("\"'")

    return url


# ================================================================================
# 3. AiPrice Amazon 상품 페이지에서 sku_id 추출
#    - SKU 추출을 최우선으로 수행
#    - 정상 HTML 내부의 일반적인 문구로 인한 차단 오탐 방지
#    - SKU 미검출 시에만 재시도
# ================================================================================
# ================================================================================
# AiPrice Amazon SKU 조회 v2
# - Cookie 누적 방지
# - HTTP 400 Request Header Or Cookie Too Large 대응
# - HTTP 오류 재시도
# ================================================================================

def resolve_aiprice_amazon_sku(amazon_url):

    amazon_url = normalize_amazon_url(amazon_url)

    if not amazon_url:
        return None, None, None

    params = {
        "link": amazon_url,
        "cateid": "0",
    }

    max_retries = 3

    for attempt in range(1, max_retries + 1):

        try:

            # --------------------------------------------------------------------
            # 핵심 수정
            # 기존 AiPrice 세션에 누적된 Cookie 제거
            # --------------------------------------------------------------------

            try:
                aiprice_session.cookies.clear()
            except Exception:
                pass

            # --------------------------------------------------------------------
            # 요청
            # --------------------------------------------------------------------

            response = aiprice_session.get(
                AIPRICE_AMAZON_URL,
                params=params,
                timeout=30,
                allow_redirects=True
            )

            status_code = response.status_code
            final_url = response.url
            html_text = response.text or ""

            # --------------------------------------------------------------------
            # HTTP 오류
            # --------------------------------------------------------------------

            if status_code != 200:

                print(
                    f"      [SKU HTTP {status_code}] "
                    f"(시도 {attempt}/{max_retries})"
                )

                if attempt < max_retries:

                    time.sleep(
                        3 * attempt
                    )

                    continue

                return (
                    None,
                    status_code,
                    final_url
                )

            # --------------------------------------------------------------------
            # 1. 정규식으로 SKU 탐색
            # --------------------------------------------------------------------

            patterns = [

                r'sku_id=([0-9]+)',

                r'["\']sku_id["\']\s*[:=]\s*["\']([0-9]+)',

                r'["\']sku_id["\']\s*[:=]\s*([0-9]+)',

                r'data-sku-id=["\']([0-9]+)',

                r'["\']skuId["\']\s*[:=]\s*["\']([0-9]+)',

                r'["\']skuID["\']\s*[:=]\s*["\']([0-9]+)',
            ]

            candidates = []

            for pattern in patterns:

                matches = re.findall(
                    pattern,
                    html_text,
                    flags=re.I
                )

                for value in matches:

                    value = str(value).strip()

                    if not value.isdigit():
                        continue

                    if len(value) < 6:
                        continue

                    if value not in candidates:
                        candidates.append(value)

            # --------------------------------------------------------------------
            # SKU 발견
            # --------------------------------------------------------------------

            if candidates:

                return (
                    candidates[0],
                    status_code,
                    final_url
                )

            # --------------------------------------------------------------------
            # 2. BeautifulSoup attribute 탐색
            # --------------------------------------------------------------------

            soup = BeautifulSoup(
                html_text,
                "html.parser"
            )

            for tag in soup.find_all(True):

                for attr_name, attr_value in tag.attrs.items():

                    attr_name_lower = str(
                        attr_name
                    ).lower()

                    if "sku" not in attr_name_lower:
                        continue

                    if isinstance(attr_value, list):

                        attr_value = " ".join(
                            map(str, attr_value)
                        )

                    value = str(
                        attr_value
                    ).strip()

                    if (
                        value.isdigit()
                        and len(value) >= 6
                    ):

                        return (
                            value,
                            status_code,
                            final_url
                        )

            # --------------------------------------------------------------------
            # SKU 미검출
            # --------------------------------------------------------------------

            html_lower = html_text.lower()

            strong_block_signals = [
                "access denied",
                "too many requests",
                "verify you are human",
                "cf-chl-",
                "cloudflare",
            ]

            block_detected = any(
                signal in html_lower
                for signal in strong_block_signals
            )

            if block_detected:

                print(
                    f"      [SKU 응답 차단 의심] "
                    f"HTTP {status_code}"
                )

            else:

                print(
                    f"      [SKU 미검출] "
                    f"HTTP {status_code}"
                )

            if attempt < max_retries:

                time.sleep(
                    3 * attempt
                )

                continue

            return (
                None,
                status_code,
                final_url
            )

        except Exception as e:

            print(
                f"      [SKU 오류 {attempt}/{max_retries}] "
                f"{type(e).__name__}: {e}"
            )

            if attempt < max_retries:

                time.sleep(
                    3 * attempt
                )

                continue

            return (
                None,
                None,
                None
            )

    return None, None, None


print("=" * 100)
print("resolve_aiprice_amazon_sku v2 교체 완료")
print("=" * 100)
print("Cookie 누적 방지 : ON")
print("HTTP 400 대응     : ON")


# ================================================================================
# 4. AiPrice 가격이력 POST
# ================================================================================

def fetch_aiprice_history(
    sku_id,
    referer=None
):

    if not sku_id:
        return [], None, None

    params = {
        "sku_id": str(sku_id),
        "adid": AIPRICE_ADID,
        "currency": AIPRICE_CURRENCY,
        "day": AIPRICE_DAY,
    }

    headers = {
        "Accept": "*/*",
        "X-Requested-With": "XMLHttpRequest",
        "Origin": AIPRICE_BASE_URL,
        "Referer": (
            referer
            if referer
            else AIPRICE_BASE_URL + "/"
        ),
    }

    try:

        response = aiprice_session.post(
            AIPRICE_HISTORY_URL,
            params=params,
            headers=headers,
            timeout=30,
            allow_redirects=True
        )

        status_code = response.status_code
        content_type = response.headers.get(
            "Content-Type",
            ""
        )

        # ------------------------------------------------------------------------
        # HTTP 확인
        # ------------------------------------------------------------------------

        if status_code != 200:

            return (
                [],
                status_code,
                content_type
            )

        # ------------------------------------------------------------------------
        # JSON 직접 파싱
        # ------------------------------------------------------------------------

        try:

            data = response.json()

        except Exception:

            # 혹시 JSON content-type이 아니어도
            # JSON 문자열이면 직접 파싱
            try:

                data = json.loads(
                    response.text
                )

            except Exception:

                return (
                    [],
                    status_code,
                    content_type
                )

        # ------------------------------------------------------------------------
        # 반드시 list 형태인지 확인
        # ------------------------------------------------------------------------

        if not isinstance(data, list):

            return (
                [],
                status_code,
                content_type
            )

        # ------------------------------------------------------------------------
        # 실제 가격이 있는 데이터만 추출
        # ------------------------------------------------------------------------

        history = []

        for item in data:

            if not isinstance(item, dict):
                continue

            raw_price = item.get("price")

            raw_timestamp = item.get(
                "time_update"
            )

            # 가격 필수
            if raw_price in (
                None,
                "",
                "0",
                "0.0",
                "0.00"
            ):
                continue

            # timestamp 필수
            if raw_timestamp in (
                None,
                "",
                0,
                "0"
            ):
                continue

            try:

                price = float(
                    str(raw_price)
                    .replace(",", "")
                    .strip()
                )

            except Exception:

                continue

            # 0 이하 가격 제거
            if price <= 0:
                continue

            try:

                timestamp = int(
                    float(raw_timestamp)
                )

                date_value = pd.to_datetime(
                    timestamp,
                    unit="s",
                    errors="coerce"
                )

            except Exception:

                continue

            if pd.isna(date_value):
                continue

            history.append({

                "가격": price,

                "날짜": date_value,

                "m_price": item.get(
                    "m_price"
                ),

                "lowprice": item.get(
                    "lowprice"
                ),

                "adid": item.get(
                    "adid"
                ),

                "src": item.get(
                    "src"
                ),

                "price_symbol": item.get(
                    "price_symbol"
                ),

                "time_update": timestamp,
            })

        # ------------------------------------------------------------------------
        # 날짜순 정렬
        # ------------------------------------------------------------------------

        history.sort(
            key=lambda x: x["날짜"]
        )

        return (
            history,
            status_code,
            content_type
        )

    except Exception as e:

        print(
            f"      [History 오류] "
            f"{type(e).__name__}: {e}"
        )

        return [], None, None


# ================================================================================
# 5. 가격이력 → 분석용 요약
# ================================================================================

def summarize_aiprice_history(history):

    if not history:
        return {
            "180일 최저가격": None,
            "180일 최고가격": None,
            "180일 평균가격": None,
            "최근 AiPrice 가격": None,
            "가격변화": None,
            "가격변화율": None,
            "가격이력건수": 0,
            "최초 가격일": pd.NaT,
            "최근 가격일": pd.NaT,
        }

    prices = [
        x["가격"]
        for x in history
        if x.get("가격") is not None
    ]

    if not prices:

        return {
            "180일 최저가격": None,
            "180일 최고가격": None,
            "180일 평균가격": None,
            "최근 AiPrice 가격": None,
            "가격변화": None,
            "가격변화율": None,
            "가격이력건수": 0,
            "최초 가격일": pd.NaT,
            "최근 가격일": pd.NaT,
        }

    prices_series = pd.Series(
        prices,
        dtype="float64"
    )

    first_price = float(
        history[0]["가격"]
    )

    latest_price = float(
        history[-1]["가격"]
    )

    price_change = (
        latest_price
        - first_price
    )

    if first_price != 0:

        price_change_rate = (
            price_change
            / first_price
            * 100
        )

    else:

        price_change_rate = None

    dates = [
        x["날짜"]
        for x in history
        if pd.notna(x["날짜"])
    ]

    return {

        "180일 최저가격":
            float(prices_series.min()),

        "180일 최고가격":
            float(prices_series.max()),

        "180일 평균가격":
            float(prices_series.mean()),

        "최근 AiPrice 가격":
            latest_price,

        "가격변화":
            price_change,

        "가격변화율":
            price_change_rate,

        "가격이력건수":
            len(history),

        "최초 가격일":
            min(dates) if dates else pd.NaT,

        "최근 가격일":
            max(dates) if dates else pd.NaT,
    }


# ================================================================================
# 6. 전체 AiPrice 처리 함수
#
# Cell 4에서 사용할 수 있도록 기존 함수 형태를 유지
# ================================================================================

def collect_aiprice_for_amazon(
    product_df
):

    print()
    print("=" * 100)
    print("AiPrice Amazon 가격이력 수집")
    print("=" * 100)

    print(
        "Amazon 상품 수 :",
        len(product_df)
    )

    # --------------------------------------------------------------------------
    # 결과용
    # --------------------------------------------------------------------------

    summary_rows = []
    history_rows = []
    quality_rows = []

    # --------------------------------------------------------------------------
    # Amazon URL 기준 중복 제거
    #
    # 같은 ASIN이 여러 페이지에 중복 노출되어도
    # AiPrice는 한 번만 조회
    # --------------------------------------------------------------------------

    work_df = product_df.copy()

    work_df["_amazon_url_key"] = (
        work_df["상품URL"]
        .astype(str)
        .str.strip()
    )

    unique_df = (
        work_df[
            work_df["_amazon_url_key"]
            .ne("")
        ]
        .drop_duplicates(
            subset=["_amazon_url_key"]
        )
        .copy()
    )

    print(
        "Unique Amazon URL 수 :",
        len(unique_df)
    )

    # --------------------------------------------------------------------------
    # 캐시
    # --------------------------------------------------------------------------

    sku_cache = {}

    history_cache = {}

    # --------------------------------------------------------------------------
    # 상품별 처리
    # --------------------------------------------------------------------------

    for idx, row in unique_df.iterrows():

        amazon_url = str(
            row.get(
                "상품URL",
                ""
            )
        ).strip()

        asin = str(
            row.get(
                "ASIN",
                ""
            )
        ).strip()

        product_name = str(
            row.get(
                "상품명",
                ""
            )
        ).strip()

        print()
        print(
            f"[{len(quality_rows)+1}/{len(unique_df)}]"
        )

        print(
            "상품명 :",
            product_name[:100]
        )

        print(
            "ASIN   :",
            asin
        )

        # ======================================================================
        # A. SKU resolve
        # ======================================================================

        if amazon_url in sku_cache:

            sku_id, sku_status, amazon_final_url = (
                sku_cache[amazon_url]
            )

        else:

            sku_id, sku_status, amazon_final_url = (
                resolve_aiprice_amazon_sku(
                    amazon_url
                )
            )

            sku_cache[amazon_url] = (
                sku_id,
                sku_status,
                amazon_final_url
            )

        if not sku_id:

            print(
                "   → SKU 실패"
            )

            quality_rows.append({

                "ASIN": asin,
                "상품명": product_name,
                "상품URL": amazon_url,

                "AiPrice SKU":
                    None,

                "연결방식":
                    "Amazon_URL",

                "SKU 요청 HTTP":
                    sku_status,

                "가격이력 HTTP":
                    None,

                "가격이력건수":
                    0,

                "상태":
                    "SKU 실패",
            })

            continue

        print(
            "   → sku_id =",
            sku_id
        )

        # ======================================================================
        # B. 가격이력
        # ======================================================================

        if sku_id in history_cache:

            history, history_status, content_type = (
                history_cache[sku_id]
            )

        else:

            history, history_status, content_type = (
                fetch_aiprice_history(
                    sku_id,
                    referer=amazon_final_url
                )
            )

            history_cache[sku_id] = (
                history,
                history_status,
                content_type
            )

        print(
            "   → 가격이력 :",
            len(history),
            "건"
        )

        # ======================================================================
        # C. 가격이력 저장
        # ======================================================================

        for h in history:

            history_rows.append({

                "ASIN":
                    asin,

                "검색키워드":
                    row.get(
                        "검색키워드",
                        ""
                    ),

                "검색순위":
                    row.get(
                        "검색순위",
                        None
                    ),

                "브랜드":
                    row.get(
                        "브랜드",
                        ""
                    ),

                "상품명":
                    product_name,

                "상품URL":
                    amazon_url,

                "AiPrice SKU":
                    sku_id,

                "날짜":
                    h["날짜"],

                "가격":
                    h["가격"],

                "이전가격":
                    None,

                "가격변화":
                    None,

                "가격변화율":
                    None,

                "m_price":
                    h.get(
                        "m_price"
                    ),

                "최저가":
                    h.get(
                        "lowprice"
                    ),

                "adid":
                    h.get(
                        "adid"
                    ),

                "src":
                    h.get(
                        "src"
                    ),

                "price_symbol":
                    h.get(
                        "price_symbol"
                    ),

                "time_update":
                    h.get(
                        "time_update"
                    ),
            })

        # ======================================================================
        # D. 가격변화 계산
        # ======================================================================

        if history:

            previous_price = None

            # 이 상품의 방금 추가된 행만 계산
            start_idx = len(
                history_rows
            ) - len(history)

            for j in range(
                start_idx,
                len(history_rows)
            ):

                current_price = (
                    history_rows[j]["가격"]
                )

                if previous_price is not None:

                    change = (
                        current_price
                        - previous_price
                    )

                    if previous_price != 0:

                        change_rate = (
                            change
                            / previous_price
                            * 100
                        )

                    else:

                        change_rate = None

                    history_rows[j][
                        "이전가격"
                    ] = previous_price

                    history_rows[j][
                        "가격변화"
                    ] = change

                    history_rows[j][
                        "가격변화율"
                    ] = change_rate

                previous_price = current_price

            # ==================================================================
            # E. 요약
            # ==================================================================

            summary = summarize_aiprice_history(
                history
            )

        else:

            summary = summarize_aiprice_history(
                []
            )

        summary_rows.append({

            "ASIN":
                asin,

            "검색키워드":
                row.get(
                    "검색키워드",
                    ""
                ),

            "검색순위":
                row.get(
                    "검색순위",
                    None
                ),

            "브랜드":
                row.get(
                    "브랜드",
                    ""
                ),

            "상품명":
                product_name,

            "상품URL":
                amazon_url,

            "AiPrice SKU":
                sku_id,

            **summary,
        })

        # ======================================================================
        # F. 품질
        # ======================================================================

        if history:

            status = "성공"

        elif history_status == 200:

            status = "가격이력 없음"

        else:

            status = "가격이력 요청 실패"

        quality_rows.append({

            "ASIN":
                asin,

            "상품명":
                product_name,

            "상품URL":
                amazon_url,

            "AiPrice SKU":
                sku_id,

            "연결방식":
                "Amazon_URL",

            "SKU 요청 HTTP":
                sku_status,

            "가격이력 HTTP":
                history_status,

            "Content-Type":
                content_type,

            "가격이력건수":
                len(history),

            "상태":
                status,
        })

        # ----------------------------------------------------------------------
        # 요청 간격
        # ----------------------------------------------------------------------

        time.sleep(
            random.uniform(
                AIPRICE_SLEEP_MIN,
                AIPRICE_SLEEP_MAX
            )
        )

    # =============================================================================
    # DataFrame
    # =============================================================================

    summary_df = pd.DataFrame(
        summary_rows
    )

    history_df = pd.DataFrame(
        history_rows
    )

    quality_df = pd.DataFrame(
        quality_rows
    )

    # =============================================================================
    # 날짜 / 숫자 타입
    # =============================================================================

    if not summary_df.empty:

        for col in [
            "최초 가격일",
            "최근 가격일",
        ]:

            if col in summary_df.columns:

                summary_df[col] = pd.to_datetime(
                    summary_df[col],
                    errors="coerce"
                )

    if not history_df.empty:

        history_df["날짜"] = pd.to_datetime(
            history_df["날짜"],
            errors="coerce"
        )

        numeric_cols = [
            "가격",
            "이전가격",
            "가격변화",
            "가격변화율",
            "m_price",
            "최저가",
        ]

        for col in numeric_cols:

            if col in history_df.columns:

                history_df[col] = pd.to_numeric(
                    history_df[col],
                    errors="coerce"
                )

    # =============================================================================
    # 최종 결과
    # =============================================================================

    print()
    print("=" * 100)
    print("AiPrice Amazon 수집 완료")
    print("=" * 100)

    print(
        "AiPrice 성공 상품 수 :",
        len(
            quality_df[
                quality_df["상태"]
                == "성공"
            ]
        )
        if not quality_df.empty
        else 0
    )

    print(
        "AiPrice 실패 상품 수 :",
        len(
            quality_df[
                quality_df["상태"]
                != "성공"
            ]
        )
        if not quality_df.empty
        else 0
    )

    print(
        "가격이력 행 수 :",
        len(history_df)
    )

    return (
        summary_df,
        history_df,
        quality_df
    )

# ================================================================================
# 7. Cell 2 전체 상품 AiPrice 수집
# ================================================================================

print()
print("=" * 100)
print("Cell 2 AiPrice 전체 수집")
print("=" * 100)

if "product_df" not in globals():

    print("❌ product_df가 없습니다.")
    print("먼저 Cell 1을 실행해주세요.")

elif product_df.empty:

    print("❌ product_df가 비어 있습니다.")

else:

    print("Amazon 전체 상품 수 :", len(product_df))

    unique_url_count = (
        product_df["상품URL"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .nunique()
    )

    print("Unique Amazon URL 수 :", unique_url_count)

    print()
    print("전체 상품 AiPrice 수집을 시작합니다.")
    print("※ 상품 수가 많아 시간이 걸릴 수 있습니다.")
    print()

    (
        price_summary_df,
        price_history_df,
        quality_df
    ) = collect_aiprice_for_amazon(
        product_df
    )

    print()
    print("=" * 100)
    print("전체 수집 결과")
    print("=" * 100)

    print()
    print("[가격요약]")
    print("행 수 :", len(price_summary_df))

    if not price_summary_df.empty:
        display(price_summary_df)

    print()
    print("[가격이력]")
    print("행 수 :", len(price_history_df))

    if not price_history_df.empty:
        display(price_history_df)

    print()
    print("[데이터 품질]")
    print("행 수 :", len(quality_df))

    if not quality_df.empty:
        display(quality_df)

    print()
    print("=" * 100)
    print("Cell 2 전체 상품 AiPrice 수집 완료")
    print("=" * 100)

**Cell 03**

In [ ]:
# ================================================================================
# ③ Amazon 제품분석 → AiPrice → Excel
#
# Cell 1
#   Amazon HTML 선택 + 상품 추출
#   → product_df 생성
#
# Cell 2
#   AiPrice 함수 정의
#
# Cell 3
#   기존 product_df 사용
#   → Cell 2 AiPrice 연결
#   → 가격요약 / 가격이력 / 데이터품질
#   → Excel 저장
#
# ※ Cell 3에서는 HTML 파일을 다시 선택하거나 다시 읽지 않음
# ================================================================================


# ================================================================================
# 0. Excel 스타일
# ================================================================================

def style_excel_sheet(ws):

    from openpyxl.styles import (
        Font,
        PatternFill,
        Alignment,
        Border,
        Side,
    )

    header_fill = PatternFill(
        "solid",
        fgColor="1F4E78",
    )

    header_font = Font(
        color="FFFFFF",
        bold=True,
    )

    thin = Side(
        style="thin",
        color="D9E2F3",
    )

    for cell in ws[1]:

        cell.fill = header_fill
        cell.font = header_font

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
        )

        cell.border = Border(
            bottom=thin
        )

    ws.freeze_panes = "A2"

    if ws.max_row >= 1:
        ws.auto_filter.ref = ws.dimensions

    for column_cells in ws.columns:

        max_length = 0

        column_letter = (
            column_cells[0].column_letter
        )

        for cell in column_cells[:300]:

            try:

                value_length = len(
                    str(cell.value)
                )

                max_length = max(
                    max_length,
                    value_length,
                )

            except Exception:
                pass

        ws.column_dimensions[
            column_letter
        ].width = min(
            max(
                max_length + 2,
                10,
            ),
            55,
        )


# ================================================================================
# 1. 현재 product_df 확인
# ================================================================================

if "product_df" not in globals():

    raise RuntimeError(
        "product_df가 없습니다.\n"
        "먼저 Cell 1에서 Amazon HTML 상품 추출을 실행해주세요."
    )

if product_df.empty:

    raise ValueError(
        "product_df가 비어 있습니다.\n"
        "먼저 Cell 1에서 Amazon 상품 추출을 실행해주세요."
    )


print("=" * 80)
print("Cell 3 시작")
print("=" * 80)

print(
    f"기존 Amazon 상품 데이터 : "
    f"{len(product_df):,}개"
)

print(
    f"Unique ASIN : "
    f"{product_df['ASIN'].nunique():,}"
)


# ================================================================================
# 2. ASIN 정상화
# ================================================================================

if "ASIN" in product_df.columns:

    product_df["ASIN"] = (
        product_df["ASIN"]
        .astype(str)
        .str.strip()
        .str.upper()
    )


# ================================================================================
# 3. Amazon Product QA
# ================================================================================

print()
print("=" * 80)
print("Amazon 제품분석 QA")
print("=" * 80)

print(
    f"전체 상품 수 : "
    f"{len(product_df):,}"
)

print()

for col in [
    "검색순위",
    "검색키워드",
    "브랜드",
    "상품명",
    "상품URL",
    "ASIN",
    "판매가격",
    "정가",
    "할인율",
    "평점",
    "리뷰수",
    "페이지",
]:

    if col in product_df.columns:

        count = (
            product_df[col]
            .notna()
            .sum()
        )

        print(
            f"{col:<12} : "
            f"{count:,} / {len(product_df):,} "
            f"({count / len(product_df) * 100:.1f}%)"
        )


# ================================================================================
# 4. Cell 2에서 이미 수집한 AiPrice 결과 사용
#
# ※ 여기서는 AiPrice 재요청하지 않음
# ※ price_summary_df
#    price_history_df
#    quality_df
#   를 그대로 사용
# ================================================================================

print()
print("=" * 80)
print("AiPrice 기존 수집 결과 사용")
print("=" * 80)

required_aiprice_objects = [
    "price_summary_df",
    "price_history_df",
    "quality_df",
]

missing_objects = [
    name
    for name in required_aiprice_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Cell 2의 AiPrice 결과가 없습니다.\n"
        f"누락된 변수 : {missing_objects}\n\n"
        "먼저 Cell 2에서 AiPrice 수집을 완료한 후 "
        "Cell 3을 실행해주세요."
    )

print(
    f"가격요약 데이터 : {len(price_summary_df):,}건"
)

print(
    f"가격이력 데이터 : {len(price_history_df):,}건"
)

print(
    f"데이터품질 데이터 : {len(quality_df):,}건"
)

print()
print("※ AiPrice 재요청하지 않습니다.")

# ================================================================================
# 5. 현재 Amazon 가격 → 가격요약
# ================================================================================

if not price_summary_df.empty:

    current_price_map = (
        product_df[
            [
                "ASIN",
                "판매가격",
            ]
        ]
        .drop_duplicates(
            subset=["ASIN"],
            keep="first",
        )
        .set_index(
            "ASIN"
        )["판매가격"]
    )

    price_summary_df[
        "현재 Amazon 가격"
    ] = (
        price_summary_df["ASIN"]
        .map(
            current_price_map
        )
    )

    summary_columns = [
        "검색순위",
        "검색키워드",
        "브랜드",
        "상품명",
        "ASIN",
        "현재 Amazon 가격",
        "180일 최저가격",
        "180일 최고가격",
        "180일 평균가격",
        "최근 AiPrice 가격",
        "가격변화",
        "가격변화율",
        "가격이력건수",
        "최초 가격일",
        "최근 가격일",
    ]

    if "가격변화" not in price_summary_df.columns:

        if (
            "최근 AiPrice 가격"
            in price_summary_df.columns
            and
            "180일 평균가격"
            in price_summary_df.columns
        ):

            price_summary_df[
                "가격변화"
            ] = (
                price_summary_df[
                    "최근 AiPrice 가격"
                ]
                -
                price_summary_df[
                    "180일 평균가격"
                ]
            )

    existing_columns = [
        col
        for col in summary_columns
        if col in price_summary_df.columns
    ]

    price_summary_df = (
        price_summary_df[
            existing_columns
        ]
    )


# ================================================================================
# 6. 가격이력 정리
# ================================================================================

if not price_history_df.empty:

    sort_columns = [
        col
        for col in [
            "검색키워드",
            "검색순위",
            "ASIN",
            "날짜",
        ]
        if col in price_history_df.columns
    ]

    if sort_columns:

        price_history_df = (
            price_history_df
            .sort_values(
                sort_columns,
                na_position="last",
            )
            .reset_index(
                drop=True
            )
        )

    if "날짜" in price_history_df.columns:

        price_history_df["날짜"] = (
            pd.to_datetime(
                price_history_df["날짜"],
                errors="coerce",
            )
        )

    for col in [
        "가격",
        "이전가격",
        "가격변화",
        "가격변화율",
        "m_price",
        "최저가",
    ]:

        if col in price_history_df.columns:

            price_history_df[col] = (
                pd.to_numeric(
                    price_history_df[col],
                    errors="coerce",
                )
            )


# ================================================================================
# 7. 데이터품질
# ================================================================================

if quality_df.empty:

    quality_df = pd.DataFrame(
        columns=[
            "ASIN",
            "상품명",
            "상품URL",
            "AiPrice SKU",
            "연결방식",
            "SKU 요청 HTTP",
            "가격이력 HTTP",
            "Content-Type",
            "가격이력건수",
            "상태",
        ]
    )


# ================================================================================
# 8. Excel 파일명
# ================================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

keywords = (
    product_df["검색키워드"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace(
        "",
        pd.NA
    )
    .dropna()
    .unique()
    .tolist()
)

if len(keywords) == 1:

    keyword_label = re.sub(
        r"[^0-9A-Za-z가-힣]+",
        "_",
        keywords[0],
    ).strip("_")

elif len(keywords) > 1:

    keyword_label = (
        f"{keywords[0]}_외{len(keywords)-1}개"
    )

else:

    keyword_label = "amazon"


output_path = os.path.join(
    OUTPUT_DIR,
    f"Amazon_{keyword_label}_AiPrice_"
    f"{timestamp}.xlsx",
)


# ================================================================================
# 9. Excel 저장
# ================================================================================

with pd.ExcelWriter(
    output_path,
    engine="openpyxl",
) as writer:

    # --------------------------------------------------------------------------
    # 제품분석
    # --------------------------------------------------------------------------

    product_df.to_excel(
        writer,
        sheet_name="제품분석",
        index=False,
    )

    # --------------------------------------------------------------------------
    # 가격요약
    # --------------------------------------------------------------------------

    price_summary_df.to_excel(
        writer,
        sheet_name="가격요약",
        index=False,
    )

    # --------------------------------------------------------------------------
    # 가격이력
    # --------------------------------------------------------------------------

    price_history_df.to_excel(
        writer,
        sheet_name="가격이력",
        index=False,
    )

    # --------------------------------------------------------------------------
    # 데이터품질
    # --------------------------------------------------------------------------

    quality_df.to_excel(
        writer,
        sheet_name="데이터품질",
        index=False,
    )

    # --------------------------------------------------------------------------
    # 스타일
    # --------------------------------------------------------------------------

    for sheet_name in [
        "제품분석",
        "가격요약",
        "가격이력",
        "데이터품질",
    ]:

        ws = writer.book[
            sheet_name
        ]

        style_excel_sheet(
            ws
        )

        if ws.max_row >= 2:

            headers = [
                ws.cell(
                    1,
                    col
                ).value
                for col in range(
                    1,
                    ws.max_column + 1
                )
            ]

            for col_idx, header in enumerate(
                headers,
                start=1,
            ):

                header_text = str(
                    header
                )

                if (
                    "날짜" in header_text
                    or "기록일" in header_text
                ):

                    for row_idx in range(
                        2,
                        ws.max_row + 1,
                    ):

                        ws.cell(
                            row_idx,
                            col_idx,
                        ).number_format = (
                            "yyyy-mm-dd"
                        )


# ================================================================================
# 10. 결과 출력
# ================================================================================

print()
print("=" * 80)
print("전체 작업 완료")
print("=" * 80)

print(
    f"Amazon 상품 수       : "
    f"{len(product_df):,}"
)

print(
    f"Unique ASIN 수       : "
    f"{product_df['ASIN'].nunique():,}"
)

if not quality_df.empty and "상태" in quality_df.columns:

    success_count = (
        quality_df[
            quality_df["상태"] == "성공"
        ].shape[0]
    )

    fail_count = (
        quality_df[
            quality_df["상태"] != "성공"
        ].shape[0]
    )

else:

    success_count = 0
    fail_count = 0


print(
    f"AiPrice 성공 상품 수 : "
    f"{success_count:,}"
)

print(
    f"AiPrice 실패 상품 수 : "
    f"{fail_count:,}"
)

print(
    f"가격이력 행 수       : "
    f"{len(price_history_df):,}"
)

print()
print(
    f"Excel 저장 위치:\n"
    f"{output_path}"
)


# ================================================================================
# 11. 샘플
# ================================================================================

print()
print("가격요약 샘플")
print("-" * 80)

if not price_summary_df.empty:

    display(
        price_summary_df.head(20)
    )


print()
print("가격이력 샘플")
print("-" * 80)

if not price_history_df.empty:

    display(
        price_history_df.head(20)
    )


print()
print("AiPrice 데이터 품질")
print("-" * 80)

display(
    quality_df.head(20)
)


# ================================================================================
# 12. 전역 저장
# ================================================================================

LAST_OUTPUT_PATH = output_path

LAST_PRODUCT_DF = product_df

LAST_PRICE_SUMMARY_DF = (
    price_summary_df
)

LAST_PRICE_HISTORY_DF = (
    price_history_df
)

LAST_QUALITY_DF = quality_df

In [ ]:
# ================================================================================
# Cell 3 - Amazon / AiPrice 결과 Excel 저장
#
# ※ HTML 파일을 다시 읽지 않음
# ※ Amazon 재수집하지 않음
# ※ AiPrice 재요청하지 않음
# ※ Cell 1 + Cell 2의 기존 결과만 사용
# ================================================================================

import os
import re
from datetime import datetime

import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, clear_output
from google.colab import files


# ================================================================================
# 1. 결과 확인
# ================================================================================

required_objects = [
    "product_df",
    "price_summary_df",
    "price_history_df",
    "quality_df",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    print("=" * 80)
    print("❌ Cell 3 실행 불가")
    print("=" * 80)
    print()
    print(f"누락된 변수 : {missing_objects}")
    print()
    print("먼저 Cell 1 → Cell 2를 실행해주세요.")

else:

    print("=" * 80)
    print("기존 Amazon / AiPrice 결과 확인")
    print("=" * 80)

    print()
    print(f"제품분석     : {len(product_df):,}건")
    print(f"가격요약     : {len(price_summary_df):,}건")
    print(f"가격이력     : {len(price_history_df):,}건")
    print(f"데이터품질   : {len(quality_df):,}건")

    print()
    print("✓ Cell 1 / Cell 2 결과 확인 완료")
    print("✓ HTML 파일을 다시 읽지 않습니다.")
    print("✓ AiPrice를 다시 요청하지 않습니다.")


# ================================================================================
# 2. 출력 폴더
# ================================================================================

AMAZON_OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "YOUR_PROJECT_FOLDER/amazon_data/amazon_output"
)

os.makedirs(AMAZON_OUTPUT_DIR, exist_ok=True)


# ================================================================================
# 3. 검색 키워드 결정
# ================================================================================

def get_amazon_keyword(df):

    if "검색키워드" in df.columns:

        values = (
            df["검색키워드"]
            .dropna()
            .astype(str)
            .str.strip()
        )

        values = values[
            (values != "") &
            (values != "검색어없음")
        ]

        if len(values) > 0:

            return values.iloc[0]

    return "amazon"


keyword_label = get_amazon_keyword(product_df)


# ================================================================================
# 4. 파일명 안전 처리
# ================================================================================

safe_keyword = re.sub(
    r'[\\/:*?"<>|]+',
    "_",
    str(keyword_label)
).strip()

if not safe_keyword:
    safe_keyword = "amazon"


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

output_path = os.path.join(
    AMAZON_OUTPUT_DIR,
    f"Amazon_{safe_keyword}_AiPrice_{timestamp}.xlsx"
)


# ================================================================================
# 5. Excel 저장 버튼
# ================================================================================

save_button = widgets.Button(
    description="📊 새 Excel 파일 생성",
    button_style="primary",
    layout=widgets.Layout(width="240px")
)

download_button = widgets.Button(
    description="⬇️ Excel 다운로드",
    disabled=True,
    layout=widgets.Layout(width="220px")
)

output_widget = widgets.Output()

LAST_OUTPUT_PATH = None


# ================================================================================
# 6. Excel 생성
# ================================================================================

def on_save(b):

    global LAST_OUTPUT_PATH

    with output_widget:

        clear_output()

        try:

            print("=" * 80)
            print("Amazon / AiPrice Excel 생성")
            print("=" * 80)

            print()
            print(f"검색키워드 : {keyword_label}")
            print(f"Amazon 상품 : {len(product_df):,}건")
            print(f"AiPrice 상품 : {len(price_summary_df):,}건")
            print(f"가격이력 : {len(price_history_df):,}건")

            # --------------------------------------------------------------------
            # DataFrame 복사
            # --------------------------------------------------------------------

            export_product_df = product_df.copy()
            export_summary_df = price_summary_df.copy()
            export_history_df = price_history_df.copy()
            export_quality_df = quality_df.copy()

            # --------------------------------------------------------------------
            # Excel 저장
            # --------------------------------------------------------------------

            print()
            print("Excel 파일 생성 중...")

            with pd.ExcelWriter(
                output_path,
                engine="openpyxl"
            ) as writer:

                export_product_df.to_excel(
                    writer,
                    sheet_name="제품분석",
                    index=False
                )

                export_summary_df.to_excel(
                    writer,
                    sheet_name="가격요약",
                    index=False
                )

                export_history_df.to_excel(
                    writer,
                    sheet_name="가격이력",
                    index=False
                )

                export_quality_df.to_excel(
                    writer,
                    sheet_name="데이터품질",
                    index=False
                )

            LAST_OUTPUT_PATH = output_path

            # --------------------------------------------------------------------
            # 완료
            # --------------------------------------------------------------------

            print()
            print("=" * 80)
            print("✅ Excel 생성 완료")
            print("=" * 80)

            print()
            print(f"파일명 : {os.path.basename(output_path)}")
            print()
            print(f"경로 : {output_path}")

            print()
            print("시트:")

            print(f"  • 제품분석   : {len(export_product_df):,}건")
            print(f"  • 가격요약   : {len(export_summary_df):,}건")
            print(f"  • 가격이력   : {len(export_history_df):,}건")
            print(f"  • 데이터품질 : {len(export_quality_df):,}건")

            download_button.disabled = False

        except Exception as e:

            print()
            print("=" * 80)
            print("❌ Excel 생성 실패")
            print("=" * 80)
            print()
            print(f"{type(e).__name__}: {e}")


# ================================================================================
# 7. 다운로드
# ================================================================================

def on_download(b):

    global LAST_OUTPUT_PATH

    if (
        LAST_OUTPUT_PATH
        and os.path.exists(LAST_OUTPUT_PATH)
    ):

        files.download(LAST_OUTPUT_PATH)

    else:

        with output_widget:

            print()
            print("❌ 먼저 '새 Excel 파일 생성' 버튼을 눌러주세요.")


# ================================================================================
# 8. 버튼 연결
# ================================================================================

save_button.on_click(on_save)
download_button.on_click(on_download)


# ================================================================================
# 9. 최종 UI
# ================================================================================

display(
    widgets.VBox([
        widgets.HTML(
            "<h4>Amazon / AiPrice Excel 결과</h4>"
        ),

        widgets.HTML(
            f"<b>검색키워드:</b> {keyword_label}"
        ),

        widgets.HTML(
            f"<b>Amazon 상품:</b> {len(product_df):,}건"
        ),

        widgets.HTML(
            f"<b>AiPrice 상품:</b> {len(price_summary_df):,}건"
        ),

        widgets.HTML(
            f"<b>가격이력:</b> {len(price_history_df):,}건"
        ),

        save_button,
        download_button,
        output_widget
    ])
)

print()
print("👉 '📊 새 Excel 파일 생성' 버튼을 눌러주세요.")